In [ ]:
using Pkg
Pkg.activate("/Users/bursche/Documents/GitHub/JPEC_BCRIT")
Base.active_project()

In [ ]:
using GeneralizedPerturbedEquilibrium
using GeneralizedPerturbedEquilibrium: Analysis
using GeneralizedPerturbedEquilibrium: InnerLayer
using GeneralizedPerturbedEquilibrium.InnerLayer: solve_inner
using GeneralizedPerturbedEquilibrium: Tearing
using ..InnerLayer
using ..InnerLayer: InnerLayerModel, solve_inner, GGJModel, GGJParameters,
    SLAYERModel, SLAYERParameters

using Plots
using Printf
default(
    fontfamily="Georgia",
    margin=12Plots.mm,
    size=(800, 500),
    dpi=150
)

In [ ]:
struct TorqueBalance{M<:InnerLayerModel,P}
    model::M
    params::P
    Q0::Float64
    P::Float64
    lu::Float64
    sval::Float64
end

function torque_balance_value(tb::TorqueBalance, Q::Number)
    Δ = solve_inner(tb.model, tb.params, ComplexF64(Q)).tearing
    alpha = 1e-2
    jxb = -imag(1.0 / (Δ + alpha))
    return 2.0 * tb.P * (tb.Q0 - Q) / jxb, Δ
end

function torque_balance_scan(tb; Qmin=-10.0, Qmax=10.0, n=20000)
    Qs = range(Qmin, Qmax; length=n)
    torque_out = [torque_balance_value(tb, q) for q in Qs]
    bal = [x[1] for x in torque_out]
    Δs = [x[2] for x in torque_out]
    positive = isfinite.(bal) .& (bal .> 0.0)

    if !any(positive)
        @warn "No positive torque balance found in the specified range. Try increasing the number of Q samples scanned or adjusting the range."
        return Qs, bal, NaN, NaN, NaN, Δs
    end

    # Find the highest point, then remove candidates within ΔQ of it.

    idx_maxima = findall(isfinite.(bal) .& (bal .> 0))
    sort!(idx_maxima, by=i -> bal[i], rev=true)
    selected = Int[]
    min_ΔQ = 0.001 * abs(Qmax - Qmin)

    for i in idx_maxima
        if all(abs(Qs[i] - Qs[j]) > min_ΔQ for j in selected)
            push!(selected, i)
        end
        length(selected) == 2 && break
    end

    if isempty(selected)
        @warn "No usable positive torque-balance maximum found. Try increasing the number of Q samples scanned or adjusting the range."
        return Qs, bal, NaN, NaN, NaN, Δs
    end

    idx_maxima = selected


    # Find all local maxima in the positive region
    #idx_positive = findall(positive)
    #idx_maxima = idx_positive[partialsortperm(bal[idx_positive], 1:min(2, length(idx_positive)); rev=true)]
    maxima = bal[idx_maxima]
    Qs_maxima = Qs[idx_maxima]
    # find index, q val and bal val of the q closest to Q_e
    idx_closest_Q_e = argmin(abs.(Qs .+ tb.params.Q_e)) # + Q_e since Q_e = -omega_e * Qconv
    q_closest_Q_e = Qs[idx_closest_Q_e]
    bal_closest_Q_e = bal[idx_closest_Q_e]

    # find index, q val and bal val of the q closest to Q_i
    idx_closest_Q_i = argmin(abs.(Qs .+ tb.params.Q_i)) # + Q_i since Q_i = -omega_i * Qconv
    q_closest_Q_i = Qs[idx_closest_Q_i]
    bal_closest_Q_i = bal[idx_closest_Q_i]
    
    println("Local maxima indices: ", idx_maxima, " with values: ", maxima, " and corresponding Qs: ", Qs_maxima, " closest Q to Q_e: ", q_closest_Q_e, " closest Q to Q_i: ", q_closest_Q_i)
    # check if 1st q_maxima is same as Q_e or Q_i from p
    if idx_maxima[1] == idx_closest_Q_e || idx_maxima[1] == idx_closest_Q_i
        if length(idx_maxima) < 2
            @warn "The first local maximum corresponds to a pole from Q_e or Q_i, but there is no second local maximum to select. Try increasing the number of Q samples scanned or adjusting the range."
            return Qs, bal, NaN, NaN, NaN, Δs
        end
        println("Warning: The first local maximum may correspond to an electron or ion diamagnetic resonance. Selecting the second local maximum instead.")
        if idx_maxima[2] == idx_closest_Q_e || idx_maxima[2] == idx_closest_Q_i
            @warn "Both the first and second local maxima correspond to poles from Q_e or Q_i. Unable to select a valid local maximum. Try increasing the number of Q samples scanned or adjusting the range."
            return Qs, bal, NaN, NaN, NaN, Δs
        end
        idx_maximum = idx_maxima[2]
        maximum = bal[idx_maximum]
        Qmaximum = Qs[idx_maximum]
    else
        idx_maximum = idx_maxima[1]
        maximum = bal[idx_maximum]
        Qmaximum = Qs[idx_maximum]
    end

    println("Selected local maximum index: ", idx_maximum, " with value: ", maximum, " and corresponding Q: ", Qmaximum)

    br_crit = sqrt(maximum / tb.lu * (tb.sval^2 / 2.0))

    return Qs, bal, Qmaximum, br_crit, idx_maximum, Δs
end

In [ ]:
p = GeneralizedPerturbedEquilibrium.InnerLayer.slayer_parameters(
    n_e=1e19, t_e=1e3, t_i=1e3,
    omega=0.0, omega_e=40, omega_i=-20,
    qval=2.0, sval_r=0.5, bt=2.0, rs=1.0, R0=3.0, mu_i=2.0, zeff=1.0,
    chi_perp=1.0, chi_tor=1.0, m=2, n=1
)
p = GeneralizedPerturbedEquilibrium.InnerLayer.SLAYERParameters(
    ising=p.ising,
    m=p.m, n=p.n,
    tau=p.tau, lu=p.lu, c_beta=p.c_beta, D_norm=p.D_norm,
    P_perp=p.P_perp, P_tor=p.P_tor,
    Q_e=2, Q_i=3, iota_e=p.iota_e,
    tauk=p.tauk, tau_r=p.tau_r, delta_n=p.delta_n,
    rs=p.rs, R0=p.R0, bt=p.bt, sval_r=p.sval_r,
    dr_val=p.dr_val, dgeo_val=p.dgeo_val,
    eta=p.eta, d_beta=p.d_beta,
    dc_tmp=p.dc_tmp, dc_type=p.dc_type
)

tb = TorqueBalance(
    GeneralizedPerturbedEquilibrium.InnerLayer.SLAYERModel(;),
    p,
    0.5,
    61.0, #1.0
    p.lu,
    p.sval_r
)

In [ ]:
Qs, bal, Qpeak, brcrit, Qpeak_ind, Δs = torque_balance_scan(tb,Qmin=-2.5, Qmax=0, n=200000)

In [ ]:
Qmax = 10
Qmin = -10
positive = isfinite.(bal) .& (bal .> 0.0)

if !any(positive)
    @warn "No positive torque balance found in the specified range. Try increasing the number of Q samples scanned or adjusting the range."
    return Qs, bal, NaN, NaN, NaN, Δs
end

# Find the highest point, then remove candidates within ΔQ of it.

idx_maxima = findall(isfinite.(bal) .& (bal .> 0))
sort!(idx_maxima, by=i -> bal[i], rev=true)
selected = Int[]
min_ΔQ = 0.001 * abs(Qmax - Qmin)

for i in idx_maxima
    if all(abs(Qs[i] - Qs[j]) > min_ΔQ for j in selected)
        push!(selected, i)
    end
    length(selected) == 2 && break
end

if isempty(selected)
    @warn "No usable positive torque-balance maximum found. Try increasing the number of Q samples scanned or adjusting the range."
    return Qs, bal, NaN, NaN, NaN, Δs
end

idx_maxima = selected


# Find all local maxima in the positive region
#idx_positive = findall(positive)
#idx_maxima = idx_positive[partialsortperm(bal[idx_positive], 1:min(2, length(idx_positive)); rev=true)]
maxima = bal[idx_maxima]
Qs_maxima = Qs[idx_maxima]
# find index, q val and bal val of the q closest to Q_e
idx_closest_Q_e = argmin(abs.(Qs .+ tb.params.Q_e)) # + Q_e since Q_e = -omega_e * Qconv
q_closest_Q_e = Qs[idx_closest_Q_e]
bal_closest_Q_e = bal[idx_closest_Q_e]

# find index, q val and bal val of the q closest to Q_i
idx_closest_Q_i = argmin(abs.(Qs .+ tb.params.Q_i)) # + Q_i since Q_i = -omega_i * Qconv
q_closest_Q_i = Qs[idx_closest_Q_i]
bal_closest_Q_i = bal[idx_closest_Q_i]

println("Local maxima indices: ", idx_maxima, " with values: ", maxima, " and corresponding Qs: ", Qs_maxima, " closest Q to Q_e: ", q_closest_Q_e, " closest Q to Q_i: ", q_closest_Q_i)
# check if 1st q_maxima is same as Q_e or Q_i from p
if idx_maxima[1] == idx_closest_Q_e || idx_maxima[1] == idx_closest_Q_i
    if length(idx_maxima) < 2
        @warn "The first local maximum corresponds to a pole from Q_e or Q_i, but there is no second local maximum to select. Try increasing the number of Q samples scanned or adjusting the range."
        return Qs, bal, NaN, NaN, NaN, Δs
    end
    println("Warning: The first local maximum may correspond to an electron or ion diamagnetic resonance. Selecting the second local maximum instead.")
    if idx_maxima[2] == idx_closest_Q_e || idx_maxima[2] == idx_closest_Q_i
        @warn "Both the first and second local maxima correspond to poles from Q_e or Q_i. Unable to select a valid local maximum. Try increasing the number of Q samples scanned or adjusting the range."
        return Qs, bal, NaN, NaN, NaN, Δs
    end
    idx_maximum = idx_maxima[2]
    maximum = bal[idx_maximum]
    Qmaximum = Qs[idx_maximum]
else
    idx_maximum = idx_maxima[1]
    maximum = bal[idx_maximum]
    Qmaximum = Qs[idx_maximum]
end

println("Selected local maximum index: ", idx_maximum, " with value: ", maximum, " and corresponding Q: ", Qmaximum)

br_crit = sqrt(maximum / tb.lu * (tb.sval^2 / 2.0))

println(Qmaximum, br_crit, idx_maximum)

In [ ]:
jxbs = [-imag(1.0 / (d + 1e-2)) for d in Δs]
T_VISC = [2.0 * tb.P * (tb.Q0 - q) for q in Qs] #(q, jxb) in zip(Qs, jxbs)]
#br_t = brcrit
#println(tb.lu, " ", tb.sval, " ", br_t, " ", p.bt, " ", 1e-2)
#@printf("br_crit = %.5e\n", brcrit)
#T_EM = tb.lu * tb.sval^2/2 * (brcrit)^2 * jxbs
T_EM = -imag(1.0 ./ (Δs .+ 1e-2)) *brcrit^2*tb.lu/(tb.sval^2/2)
i = Qpeak_ind
println(T_VISC[i]/(tb.lu * tb.sval^2/2 * (brcrit)^2 * jxbs[i]))
println(T_VISC[i] / (tb.lu * tb.sval^2/2 * (brcrit * p.bt)^2 * jxbs[i]))
println(T_VISC[i] / ( (brcrit / p.bt)^2 * jxbs[i]))

#T_EM ∝ S ξ̂ (br/Bφ)² Im[-Δ̂(Q)⁻¹]
#T_visc ∝ 2 P (Q0 − Q)

xmi = -2.5
xma = 0

"""
p1 = plot(Qs, imag.(Δs), label="Im(Δ)", lw=2, xlim=(xmi, xma))
#plot!(p1, Qs, real.(Δs), label="Re(Δ)", lw=2)
xlabel!(p1, "Q")
ylabel!(p1, "Δ")
#title!(p1, "Inner-layer Δ(Q)")
"""

p1 = plot( Qs, T_VISC, label="T_V", lw=2, xlim=(xmi, xma))#,ylim=(-1000,1000))
plot!(p1, Qs, T_EM, label="T_EM", lw=2)
vline!([Qpeak], label="peak Q", linestyle=:dash)
xlabel!(p1, "Q")
ylabel!(p1, "Torque")

#p2 = plot(Qs, jxbs, label="jxb", lw=2)
p2 = plot( Qs, T_VISC, label="T_V", lw=2, xlim=(xmi, xma))#,ylim=(0,500))
plot!(p2, Qs, T_EM, label="T_EM", lw=2)
vline!([Qpeak], label="peak Q", linestyle=:dash)
xlabel!(p2, "Q")
ylabel!(p2, "Torque")
#title!(p2, "jxb(Q) = -Im[1/(Δ + δ_n_p)]")

p3 = plot(Qs, real.(bal), label="Re(balance)", lw=2, xlim=(xmi, xma)) #xlim=(-1.3,-.8),ylim=(375,389))#
plot!(p3, Qs, imag.(bal), label="Im(balance)", lw=2)
loc_max_ind= [4001, 10050, 10094, 10186, 10195, 10213, 10258, 10272, 10340, 10355, 10365, 10376, 10385, 10387, 10392, 10405, 10412, 10421, 10430, 10443, 10450, 10457, 10470, 10473, 10494, 10498, 10512, 10520, 10537, 10544, 10552, 10557, 10573, 10576, 10578, 10586, 10588, 10596, 10606, 10610, 10632, 10637, 10648, 10655, 10659, 10665, 10670, 10676, 10679, 10691, 10695, 10697, 10705, 10707, 10719, 10730, 10742, 10746, 10751, 10755, 10761, 10765, 10770, 10776, 10778, 10783, 10786, 10788, 10793, 10798, 10810, 10814, 10821, 10824, 10829, 10834, 10837, 10841, 10843, 10847, 10849, 10856, 10858, 10861, 10863, 10868, 10871, 10877, 10881, 10887, 10892, 10895, 10897, 10900, 10904, 10906, 10909, 10911, 10916, 10918, 10923, 10929, 10931, 10933, 10937, 10942, 10945, 10950, 10955, 10959, 10963, 10969, 10971, 10974, 10976, 10981, 10984, 10988, 10991, 10994, 10997, 11001, 11003, 11005, 11012, 11015, 11018, 11021, 11024, 11027, 11029, 11031, 11033, 11036, 11039, 11042, 11044, 11048, 11053, 11056, 11058, 11060, 11062, 11064, 11068, 11078, 11080, 11083, 11085, 11096, 11098, 11100, 11102, 11104, 11106, 11109, 11111, 11116, 11119, 11121, 11128, 11131, 11133, 11139, 11145, 11148, 11150, 11152, 11155, 11157, 11162, 11165, 11169, 11174, 11180, 11183, 11186, 11188, 11205, 11218, 11220, 11222, 11224, 11228, 11233, 11235, 11243, 11249, 11254, 11257, 11259, 11276, 11281, 11294, 11299, 11303, 11306, 11312, 11316, 11319, 11322, 11335, 11341, 11358, 11369, 11376, 11381, 11386, 11391, 11394, 11396, 11399, 11406, 11410, 11417, 11420, 11422, 11428, 11437, 11446, 11448, 11453, 11457, 11469, 11473, 11486, 11497, 11513, 11540, 11542, 11544, 11552, 11554, 11565, 11570, 11573, 11578, 11587, 11594, 11597, 11602, 11614, 11620, 11633, 11654, 11660, 11662, 11669, 11677, 11682, 11687, 11711, 11764, 11804, 11806, 11821, 11848, 11860, 11882, 11901, 11928, 11991, 12108]
#plot!(p3, Qs[loc_max_ind], bal[loc_max_ind], seriestype=:scatter, label="Local Maxima", color=:red)
loc_mx2 = [4001, 10963, 10955, 10950, 11005]
#plot!(p3, Qs[loc_mx2], bal[loc_mx2], seriestype=:scatter, label="Local Maxima 2", color=:green)
ylabel!(p3, "Torque Balance")
#title!(p3, "2P(Q0-Q)/jxb")
xlabel!(p3, "Q")
vline!([Qpeak], label="Max Balance", linestyle=:dash)

# find index, q val and bal val of the q closest to Q_e
println("Q_e: ", -tb.params.Q_e)
idx_closest_Q_e = argmin(abs.(Qs .+ tb.params.Q_e))
q_closest_Q_e = Qs[idx_closest_Q_e]
bal_closest_Q_e = bal[idx_closest_Q_e]
println("Closest Q to Q_e: ", q_closest_Q_e, " with balance value: ", bal_closest_Q_e, " at index: ", idx_closest_Q_e)
println("B crit: ", brcrit)
vline!([-tb.params.Q_e], label="Q_e", linestyle=:dashdot, color=:orange)
vline!([q_closest_Q_e], label="Closest Q to Q_e", linestyle=:dashdot, color=:purple)

#plot(p1, p2, p3, layout=(3,1), size=(800, 1000))
plot(p3, layout=(1,1), size=(800, 1000))

In [ ]:
# compare Q_e to Q of max(bal)
println("-Q_e = ", -p.Q_e, " Q_i = ", p.Q_i, " Q_peak = ", Qpeak, " br_crit = ", brcrit)
# check if |Qpeak+Q_e| < 1e-2
println("|Qpeak + Q_e| = ", abs(Qpeak + p.Q_e))


In [ ]:
# Check torque is balanced

viscous_torque = 2*tb.P*(tb.Q0 - Qpeak)
electromagnetic_torque = -imag(1.0 / (Δs[Qpeak_ind] + 1e-2)) *brcrit^2*tb.lu/(tb.sval^2/2)

# delta_n_p = 1e-2
# jxb = -imag(1.0 / (Δ + delta_n_p))
# bal = 2.0 * tb.P * (tb.Q0 - Q) / jxb
# br_crit = sqrt(maxbal / tb.lu * (tb.sval^2 / 2.0))

@printf("br_crit = %.5e\n", brcrit)
@printf("viscous_torque = %.5e\n", viscous_torque)
@printf("electromagnetic_torque = %.5e\n", electromagnetic_torque)
@printf("ratio = %.10f\n", viscous_torque / electromagnetic_torque)